<a href="https://colab.research.google.com/github/JuanAntonioLeonOjeda/03MIAR---Algoritmos-de-Optimizacion/blob/main/Copia_de_Trabajo_Pr%C3%A1ctico_Algoritmos.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Algoritmos de optimización - Trabajo Práctico<br>
Nombre y Apellidos: Juan Antonio León Ojeda  <br>
Url: https://github.com/JuanAntonioLeonOjeda/03MIAR---Algoritmos-de-Optimizacion/tree/main/TRABAJO_PRACTICO<br>
Google Colab: https://colab.research.google.com/drive/17i9e51u3h6eMk9lQVqFDdGxHE9kPpins <br>
Problema:
>3. Configuración de Tribunales

Descripción del problema:

Se precisa configurar tribunales de evaluación para un grupo de 15 alumnos que desean
presentar su Trabajo Fin de Máster (TFM).
Cada tribunal está compuesto por tres profesores, cada uno desempeñando uno de los
siguientes roles: Presidente, Secretario o Vocal.
Los profesores han indicado su disponibilidad horaria para participar en los tribunales de
15h a 21h durante la semana del 15 al 19 de abril: <br>
Número de profesores : 10 <br>
Número de tribunales : 15 <br>
Hay 15 alumnos, por lo que se deben configurar 15 tribunales buscando la configuración
más equilibrada posible en cuanto a la cantidad de tribunales asignados a cada profesor, es
decir, evitando que un profesor tenga muchos tribunales y otros pocos.
Obviamente ningún profesor puede asistir a dos tribunales a la misma fecha/hora y no
puede ser convocado a un tribunal al que no tiene disponibilidad.









                                        

#Modelo

- ¿Como represento el espacio de soluciones?

In [2]:
# El espacio de soluciones será un array de diccionarios (podría usarse simplemente arrays anidados, pero los diccionarios proporcionarán una mayor claridad de cada asignación).
# El formato será como el que sigue:
'''
solucion = [
    {
        t: 0,      # tribunal
        d: 15,     # día
        h: 16,     # hora
        p: 'ABC',  # presidente
        s: 'DEF',  # secretario
        v: 'GHI'.  # vocal
    },
    ...
]
'''

"\nsolucion = [\n    {\n        t: 0,      # tribunal\n        d: 15,     # día\n        h: 16,     # hora\n        p: 'ABC',  # presidente\n        s: 'DEF',  # secretario\n        v: 'GHI'.  # vocal\n    },\n    ...\n]\n"

- ¿Cual es la función objetivo?

El objetivo principal del modelo tiene que ser alcanzar un reparto equilibrado de asignaciones de tribunales para cada profesor.
Para ello, se calculará la desviación estándar para medir la dispersión de los valores (cantidad de asignaciones) respecto a la media.
La función objetivo tiene que intentar que esta desviación estándar sea la menor posible para cada profesor, a través de la siguiente fórmula

σ = √[Σ(xi - x̄)² / n]

Donde:
xi: número de tribulaes asignados al profesor i
x̄: media de asignaciones
n: total de profesores

- ¿Como implemento las restricciones?

In [3]:
# Las restricciones serán un conjunto de arrays y de diccionarios. Básicamente se usarán arrays para listar los elementos por los que hay que iterar (profesores, días y horas) y diccionarios para hacer consultas (disponibilidad y roles de cada profesro)

profesores = ['RRD', 'QYV', 'LHL', 'HLC', 'MSB', 'PMQ', 'QWF', 'EBB', 'IOE', 'IOA']
dias = [15, 16, 17, 18, 19]
horas = [15, 16, 17, 18, 19, 20, 21]
disponibilidad = {
    'RRD': { # Revisado
        (15, 15): 0, (15, 16): 1, (15, 17): 1, (15, 18): 1, (15, 19): 0, (15, 20): 1, (15, 21): 1,
        (16, 15): 1, (16, 16): 0, (16, 17): 1, (16, 18): 1, (16, 19): 1, (16, 20): 1, (16, 21): 1,
        (17, 15): 1, (17, 16): 1, (17, 17): 0, (17, 18): 0, (17, 19): 1, (17, 20): 0, (17, 21): 1,
        (18, 15): 0, (18, 16): 1, (18, 17): 1, (18, 18): 1, (18, 19): 1, (18, 20): 1, (18, 21): 1,
        (19, 15): 1, (19, 16): 1, (19, 17): 1, (19, 18): 1, (19, 19): 1, (19, 20): 0, (19, 21): 0,
    },
    'QYV': { # Revisado
        (15, 15): 1, (15, 16): 1, (15, 17): 1, (15, 18): 1, (15, 19): 0, (15, 20): 0, (15, 21): 0,
        (16, 15): 0, (16, 16): 1, (16, 17): 1, (16, 18): 1, (16, 19): 1, (16, 20): 0, (16, 21): 0,
        (17, 15): 1, (17, 16): 0, (17, 17): 0, (17, 18): 1, (17, 19): 1, (17, 20): 1, (17, 21): 0,
        (18, 15): 1, (18, 16): 1, (18, 17): 1, (18, 18): 1, (18, 19): 1, (18, 20): 1, (18, 21): 1,
        (19, 15): 1, (19, 16): 1, (19, 17): 1, (19, 18): 1, (19, 19): 1, (19, 20): 1, (19, 21): 1,
    },
    'LHL': { #Revisado
        (15, 15): 0, (15, 16): 0, (15, 17): 1, (15, 18): 1, (15, 19): 0, (15, 20): 1, (15, 21): 1,
        (16, 15): 1, (16, 16): 1, (16, 17): 1, (16, 18): 0, (16, 19): 0, (16, 20): 1, (16, 21): 1,
        (17, 15): 1, (17, 16): 1, (17, 17): 1, (17, 18): 1, (17, 19): 1, (17, 20): 1, (17, 21): 1,
        (18, 15): 1, (18, 16): 0, (18, 17): 1, (18, 18): 1, (18, 19): 1, (18, 20): 0, (18, 21): 1,
        (19, 15): 0, (19, 16): 1, (19, 17): 1, (19, 18): 0, (19, 19): 1, (19, 20): 0, (19, 21): 1,
    },
    'HLC': { # Revisado
        (15, 15): 1, (15, 16): 0, (15, 17): 1, (15, 18): 0, (15, 19): 1, (15, 20): 1, (15, 21): 0,
        (16, 15): 1, (16, 16): 0, (16, 17): 0, (16, 18): 1, (16, 19): 1, (16, 20): 1, (16, 21): 1,
        (17, 15): 0, (17, 16): 0, (17, 17): 1, (17, 18): 1, (17, 19): 1, (17, 20): 1, (17, 21): 1,
        (18, 15): 1, (18, 16): 0, (18, 17): 1, (18, 18): 1, (18, 19): 0, (18, 20): 1, (18, 21): 1,
        (19, 15): 1, (19, 16): 1, (19, 17): 1, (19, 18): 1, (19, 19): 1, (19, 20): 1, (19, 21): 0,
    },
    'MSB': { # Revisado
        (15, 15): 1, (15, 16): 1, (15, 17): 0, (15, 18): 1, (15, 19): 0, (15, 20): 1, (15, 21): 1,
        (16, 15): 1, (16, 16): 1, (16, 17): 1, (16, 18): 0, (16, 19): 1, (16, 20): 1, (16, 21): 1,
        (17, 15): 1, (17, 16): 0, (17, 17): 1, (17, 18): 1, (17, 19): 0, (17, 20): 1, (17, 21): 1,
        (18, 15): 0, (18, 16): 1, (18, 17): 1, (18, 18): 1, (18, 19): 0, (18, 20): 1, (18, 21): 1,
        (19, 15): 1, (19, 16): 0, (19, 17): 1, (19, 18): 1, (19, 19): 1, (19, 20): 1, (19, 21): 0,
    },
    'PMQ': { # Revisado
        (15, 15): 1, (15, 16): 1, (15, 17): 1, (15, 18): 1, (15, 19): 1, (15, 20): 0, (15, 21): 0,
        (16, 15): 1, (16, 16): 1, (16, 17): 1, (16, 18): 1, (16, 19): 1, (16, 20): 1, (16, 21): 1,
        (17, 15): 1, (17, 16): 1, (17, 17): 0, (17, 18): 0, (17, 19): 1, (17, 20): 1, (17, 21): 1,
        (18, 15): 1, (18, 16): 1, (18, 17): 1, (18, 18): 0, (18, 19): 0, (18, 20): 1, (18, 21): 1,
        (19, 15): 1, (19, 16): 1, (19, 17): 1, (19, 18): 0, (19, 19): 1, (19, 20): 0, (19, 21): 1,
    },
    'QWF': { # Revisado
        (15, 15): 0, (15, 16): 1, (15, 17): 1, (15, 18): 1, (15, 19): 1, (15, 20): 1, (15, 21): 1,
        (16, 15): 1, (16, 16): 1, (16, 17): 0, (16, 18): 1, (16, 19): 1, (16, 20): 0, (16, 21): 1,
        (17, 15): 0, (17, 16): 0, (17, 17): 1, (17, 18): 1, (17, 19): 0, (17, 20): 0, (17, 21): 1,
        (18, 15): 1, (18, 16): 0, (18, 17): 0, (18, 18): 0, (18, 19): 0, (18, 20): 1, (18, 21): 1,
        (19, 15): 1, (19, 16): 1, (19, 17): 1, (19, 18): 1, (19, 19): 1, (19, 20): 0, (19, 21): 1,
    },
    'EBB': { # Revisado
        (15, 15): 1, (15, 16): 1, (15, 17): 1, (15, 18): 1, (15, 19): 1, (15, 20): 0, (15, 21): 0,
        (16, 15): 1, (16, 16): 1, (16, 17): 0, (16, 18): 1, (16, 19): 1, (16, 20): 1, (16, 21): 0,
        (17, 15): 1, (17, 16): 1, (17, 17): 1, (17, 18): 0, (17, 19): 0, (17, 20): 1, (17, 21): 1,
        (18, 15): 0, (18, 16): 1, (18, 17): 1, (18, 18): 1, (18, 19): 1, (18, 20): 1, (18, 21): 1,
        (19, 15): 0, (19, 16): 1, (19, 17): 1, (19, 18): 1, (19, 19): 0, (19, 20): 1, (19, 21): 0,
    },
    'IOE': { # Revisado
        (15, 15): 1, (15, 16): 0, (15, 17): 1, (15, 18): 1, (15, 19): 0, (15, 20): 1, (15, 21): 0,
        (16, 15): 0, (16, 16): 1, (16, 17): 1, (16, 18): 1, (16, 19): 1, (16, 20): 1, (16, 21): 1,
        (17, 15): 1, (17, 16): 1, (17, 17): 0, (17, 18): 0, (17, 19): 0, (17, 20): 1, (17, 21): 1,
        (18, 15): 1, (18, 16): 1, (18, 17): 1, (18, 18): 1, (18, 19): 1, (18, 20): 1, (18, 21): 0,
        (19, 15): 1, (19, 16): 0, (19, 17): 1, (19, 18): 1, (19, 19): 1, (19, 20): 1, (19, 21): 1,
    },
    'IOA': { # Revisado
        (15, 15): 1, (15, 16): 1, (15, 17): 0, (15, 18): 1, (15, 19): 1, (15, 20): 0, (15, 21): 1,
        (16, 15): 1, (16, 16): 0, (16, 17): 0, (16, 18): 0, (16, 19): 0, (16, 20): 0, (16, 21): 1,
        (17, 15): 1, (17, 16): 1, (17, 17): 0, (17, 18): 0, (17, 19): 1, (17, 20): 1, (17, 21): 1,
        (18, 15): 1, (18, 16): 0, (18, 17): 0, (18, 18): 1, (18, 19): 1, (18, 20): 1, (18, 21): 1,
        (19, 15): 1, (19, 16): 1, (19, 17): 1, (19, 18): 0, (19, 19): 0, (19, 20): 0, (19, 21): 1,
    },
}

roles = {
  'RRD': ['P', 'S', 'V'],
  'QYV': ['P', 'S', 'V'],
  'LHL': ['P', 'V'],
  'HLC': ['S', 'V'],
  'MSB': ['P', 'S', 'V'],
  'PMQ': ['P', 'S', 'V'],
  'QWF': ['S', 'V'],
  'EBB': ['S', 'V'],
  'IOE': ['P', 'S', 'V'],
  'IOA': ['P', 'S', 'V'],
}

#Análisis
- ¿Que complejidad tiene el problema?. Orden de complejidad y Contabilizar el espacio de soluciones

In [4]:
#Respuesta


#Diseño
- ¿Que técnica utilizo? ¿Por qué?

He empleado la técnica de Ramificación y Poda, ya que se trata una función objetivo para optimización y con restricciones específicas. Además, el problema trabaja solo con números enteros (1 y 0), lo cual encaja con esta técnica.

# Resolución

In [14]:
# Preparar datos
# Listas de profesores aptos para cada rol, para no tener que hacer la comprobación en cada iteración
presidentes = [ p for p in profesores if 'P' in roles[p] ]
secretarios = [ p for p in profesores if 'S' in roles[p] ]
vocales = [ p for p in profesores if 'V' in roles[p] ]

# Todos los turnos disponibles para consultar disponibilidad más tarde
turnos = [
  (15, 15), (15, 16), (15, 17), (15, 18), (15, 19), (15, 20), (15, 21), # día 15
  (16, 15), (16, 16), (16, 17), (16, 18), (16, 19), (16, 20), (16, 21), # día 16
  (17, 15), (17, 16), (17, 17), (17, 18), (17, 19), (17, 20), (17, 21), # día 17
  (18, 15), (18, 16), (18, 17), (18, 18), (18, 19), (18, 20), (18, 21), # día 18
  (19, 15), (19, 16), (19, 17), (19, 18), (19, 19), (19, 20), (19, 21)  # día 19
]

# Prefiltrado previo, donde hacemos una lista de profesores disponibles para cada turno
def revisar_disponibilidad():
  resultado = {}
  for turno in turnos:
      resultado[turno] = [ p for p in profesores if disponibilidad[p][turno] ]
  return resultado

# Crear diccionario con los profesores disponibes para cada día y hora
disp = revisar_disponibilidad()



#---------------------------------------------------------------------
#Inicialización variables
mejor_solucion = None
cota_superior = float('inf') # desviación típica inicial de las cargas de trabajo de los profesores
solucion_actual = []
tribunales_ocupados = {}
carga_profesores = {p: 0 for p in profesores}
max = 15 # número total de tribunales a asignar

# Ordenar los turnos de mayor a menor cantidad de profesores disponibles, y seleccionar los 15 primeros turnos
turnos_por_disponibilidad = []
for slot in turnos:
  dia, hora = slot
  disponibles = len(disp[(dia, hora)])
  turnos_por_disponibilidad.append((slot, disponibles))

turnos_por_disponibilidad.sort(key=lambda x: x[1], reverse=True)

turnos_seleccionados = [turno for turno, _ in turnos_por_disponibilidad[:max]] # 15 turnos con mayor disponibilidad de profesores

#---------------------------------------------------------------------
# Funciones usadas durante el ejercicio

def calcular_desv_tipica():
  total_profesores = len(profesores)
  valores = list(carga_profesores.values())
  promedio = sum(valores) / total_profesores
  varianza = sum((v - promedio)**2 for v in valores) / total_profesores
  return varianza ** 0.5

def esta_ocupado(p, d, h):
  return (d, h) in tribunales_ocupados and p in tribunales_ocupados[(d, h)]

def asignar_tribunal(n, dia, hora, pres, sec, voc):
  solucion_actual.append({'tribunal': n, 'dia': dia, 'hora': hora, 'P': pres, 'S': sec, 'V': voc})
  if (dia, hora) not in tribunales_ocupados:
    tribunales_ocupados[(dia, hora)] = []
  for prof in [pres, sec, voc]:
    tribunales_ocupados[(dia, hora)].append(prof)
    carga_profesores[prof] += 1

def deshacer_tribunal():
  if not solucion_actual:
    return
  t = solucion_actual.pop()
  for prof in [t['P'], t['S'], t['V']]:
    carga_profesores[prof] -= 1
    tribunales_ocupados[(t['dia'], t['hora'])].remove(prof)
  if not tribunales_ocupados[(t['dia'], t['hora'])]:
    del tribunales_ocupados[(t['dia'], t['hora'])]

def calcular_cota_inferior():
  tribunales_asignados = len(solucion_actual)
  tribunales_restantes = max - tribunales_asignados

  if tribunales_restantes == 0:
    return calcular_desv_tipica()

  # Distribuir restantes de forma perfecta
  asignaciones_restantes = tribunales_restantes * 3
  cargas_proyectadas = carga_profesores.copy()

  for _ in range(asignaciones_restantes):
    profesor_menos_cargado = min(cargas_proyectadas, key=cargas_proyectadas.get)
    cargas_proyectadas[profesor_menos_cargado] += 1

  valores = list(cargas_proyectadas.values())
  promedio = sum(valores) / len(valores)
  varianza = sum((c - promedio)**2 for c in valores) / len(valores)
  return varianza ** 0.5

def generar_tribunales(idx_turno):
  global mejor_solucion, cota_superior

  # Caso base: solución completa
  if idx_turno >= len(turnos_seleccionados):
    desbalance = calcular_desv_tipica()

    # Actualizar cota superior si encontramos mejor
    if desbalance < cota_superior:
      cota_superior = desbalance  # ← Actualizar cota superior
      mejor_solucion = [t.copy() for t in solucion_actual]
      print(f"✓ Nueva cota superior: {cota_superior:.4f}")
    return

  # PODA: Comparar cota inferior vs cota superior
  cota_inferior = calcular_cota_inferior()
  if cota_inferior >= cota_superior:
    # Esta rama no puede mejorar la cota superior
    return

  dia, hora = turnos_seleccionados[idx_turno]
  disponibles = set(disp[(dia, hora)])
  libres = {p for p in disponibles if not esta_ocupado(p, dia, hora)}

  cand_P = libres & set(presidentes)
  cand_S = libres & set(secretarios)
  cand_V = libres & set(vocales)

  if cand_P and cand_S and cand_V:
    # Ordenar por carga (menor primero)
    cand_P_ordenados = sorted(cand_P, key=lambda p: carga_profesores[p])
    cand_S_ordenados = sorted(cand_S, key=lambda p: carga_profesores[p])
    cand_V_ordenados = sorted(cand_V, key=lambda p: carga_profesores[p])

    for pres in cand_P_ordenados:
      for sec in cand_S_ordenados:
        if sec == pres:
          continue
        for voc in cand_V_ordenados:
          if voc in [pres, sec]:
            continue

          asignar_tribunal(idx_turno, dia, hora, pres, sec, voc)
          generar_tribunales(idx_turno + 1)
          deshacer_tribunal()

generar_tribunales(0)

print("\n" + "="*60)
print("RESULTADO FINAL")
print("="*60)
if mejor_solucion:
  print(f"Desbalance final: {cota_superior:.4f}")

  cargas_finales = {p: 0 for p in profesores}
  for t in mejor_solucion:
    cargas_finales[t['P']] += 1
    cargas_finales[t['S']] += 1
    cargas_finales[t['V']] += 1

  print("\nCargas:")
  for prof, carga in sorted(cargas_finales.items(), key=lambda x: x[1], reverse=True):
    print(f"  {prof}: {carga}")

  print("\nTribunales:")
  for t in sorted(mejor_solucion, key=lambda x: x['tribunal']):
    print(f"  T{t['tribunal']}: Día {t['dia']}, {t['hora']}:00 - P:{t['P']}, S:{t['S']}, V:{t['V']}")
else:
  print("No se encontró solución")
'''
def asignar_tribunal(n, d, hh, p, s, v):
  solucion_actual.append({'tribunal': n, 'dia': d, 'hora': hh, 'P': p, 'S': s, 'V': v})
  if (d, h) not in ocupados:
    ocupados[(d, h)] = []
  for prof in [p, s, v]:
    ocupados[(d, h)].append(prof)
    cargas[prof] += 1

def revisar_roles(set_p, set_s, set_v):
  # Revisar que hay 3 profes distintos, uno para cada rol
  for p in set_p:
    for s in set_s:
      if s == p:
        continue
      for v in set_v:
        if v in [p, s]:
          continue
        return True
  return False

def revisar_horas(d, h, n, solucion):
  if h > horas[-1]:
    return

  print(f'día:{d}, hora: {h}')
  print(disp[(d,h)])

  profesores_disponibles = set(disp[(d,h)])

  candidatos_p = profesores_disponibles & set(presidentes)
  candidatos_s = profesores_disponibles & set(secretarios)
  candidatos_v = profesores_disponibles & set(vocales)

  es_valido = revisar_roles(candidatos_p, candidatos_s, candidatos_v)

  if es_valido:
    print('de locos')
  else:
    revisar_horas(d, h + 1, n, solucion)

def revisar_dia(d, n, solucion):
  if d > dias[-1]:
    return

  # Revisar las horas del día
  revisar_horas(d, horas[0], n, solucion)
  # Una vez acabadas las horas del día, pasamos al siguiente día
  revisar_dia(d + 1, n, solucion)

def generar_tribunal(n, solucion):
  if n > max:
    return solucion

  # Crear nuevo objeto en el array de solución, para el tribunal de esta iteración. Iremos añadiendo el resto de información
  solucion.append({
      'tribunal': n
  })

  revisar_dia(dias[0], n, solucion)

  return generar_tribunal(n+1, solucion)

def start():
  tribunales_asignados = 0


start()
'''

✓ Nueva cota superior: 0.5000

RESULTADO FINAL
Desbalance final: 0.5000

Cargas:
  RRD: 5
  LHL: 5
  PMQ: 5
  EBB: 5
  IOA: 5
  QYV: 4
  HLC: 4
  MSB: 4
  QWF: 4
  IOE: 4

Tribunales:
  T0: Día 19, 17:00 - P:PMQ, S:HLC, V:EBB
  T1: Día 15, 18:00 - P:RRD, S:MSB, V:LHL
  T2: Día 17, 21:00 - P:IOA, S:QWF, V:IOE
  T3: Día 18, 20:00 - P:QYV, S:PMQ, V:EBB
  T4: Día 18, 21:00 - P:RRD, S:MSB, V:LHL
  T5: Día 15, 17:00 - P:IOE, S:QYV, V:QWF
  T6: Día 16, 15:00 - P:IOA, S:HLC, V:PMQ
  T7: Día 16, 19:00 - P:RRD, S:EBB, V:MSB
  T8: Día 16, 21:00 - P:LHL, S:IOA, V:QWF
  T9: Día 17, 15:00 - P:IOE, S:QYV, V:PMQ
  T10: Día 17, 20:00 - P:MSB, S:HLC, V:EBB
  T11: Día 18, 17:00 - P:RRD, S:QYV, V:LHL
  T12: Día 18, 18:00 - P:IOA, S:HLC, V:IOE
  T13: Día 19, 15:00 - P:PMQ, S:QWF, V:RRD
  T14: Día 19, 16:00 - P:LHL, S:EBB, V:IOA


"\ndef asignar_tribunal(n, d, hh, p, s, v):\n  solucion_actual.append({'tribunal': n, 'dia': d, 'hora': hh, 'P': p, 'S': s, 'V': v})\n  if (d, h) not in ocupados:\n    ocupados[(d, h)] = []\n  for prof in [p, s, v]:\n    ocupados[(d, h)].append(prof)\n    cargas[prof] += 1\n\ndef revisar_roles(set_p, set_s, set_v):\n  # Revisar que hay 3 profes distintos, uno para cada rol\n  for p in set_p:\n    for s in set_s:\n      if s == p:\n        continue\n      for v in set_v:\n        if v in [p, s]:\n          continue\n        return True \n  return False\n\ndef revisar_horas(d, h, n, solucion):\n  if h > horas[-1]:\n    return\n\n  print(f'día:{d}, hora: {h}')\n  print(disp[(d,h)])\n\n  profesores_disponibles = set(disp[(d,h)])\n\n  candidatos_p = profesores_disponibles & set(presidentes)\n  candidatos_s = profesores_disponibles & set(secretarios)\n  candidatos_v = profesores_disponibles & set(vocales)\n\n  es_valido = revisar_roles(candidatos_p, candidatos_s, candidatos_v)\n\n  if es_val

In [ ]:
import time

max_tribunales = 15
presidentes = [p for p in profesores if 'P' in roles[p]]
secretarios = [p for p in profesores if 'S' in roles[p]]
vocales = [p for p in profesores if 'V' in roles[p]]

presidentes_set = set(presidentes)
secretarios_set = set(secretarios)
vocales_set = set(vocales)

# CLAVE 1: Pre-seleccionar slots (no 35, solo ~18-20)
slots_buenos = []
for d in dias:
  for h in horas:
    disponibles = len(disp[(d, h)])
    if disponibles >= 5:  # Solo slots con buena disponibilidad
      slots_buenos.append((d, h))

print(f"Slots seleccionados: {len(slots_buenos)}")

mejor_solucion = None
mejor_desbalance = float('inf')
solucion_actual = []
ocupados = {}
cargas = {p: 0 for p in profesores}
nodos_explorados = 0

def esta_ocupado(profesor, dia, hora):
  if (dia, hora) in ocupados:
    return profesor in ocupados[(dia, hora)]
  return False

def asignar_tribunal(n, dia, hora, pres, sec, voc):
  solucion_actual.append({
    'tribunal': n, 'dia': dia, 'hora': hora,
    'P': pres, 'S': sec, 'V': voc
  })
  if (dia, hora) not in ocupados:
    ocupados[(dia, hora)] = []
  for prof in [pres, sec, voc]:
    ocupados[(dia, hora)].append(prof)
    cargas[prof] += 1

def deshacer_tribunal():
  if not solucion_actual:
    return
  t = solucion_actual.pop()
  for prof in [t['P'], t['S'], t['V']]:
    cargas[prof] -= 1
    ocupados[(t['dia'], t['hora'])].remove(prof)
  if not ocupados[(t['dia'], t['hora'])]:
    del ocupados[(t['dia'], t['hora'])]

def calcular_desbalance():
  valores = list(cargas.values())
  promedio = sum(valores) / len(valores)
  varianza = sum((c - promedio)**2 for c in valores) / len(valores)
  return varianza ** 0.5

def generar_tribunales(idx_slot, tribunales_asignados):
  global mejor_solucion, mejor_desbalance, nodos_explorados

  nodos_explorados += 1

  # Progreso
  if nodos_explorados % 5000 == 0:
    print(f"Nodos: {nodos_explorados}, Mejor: {mejor_desbalance:.4f}, Tiempo: {time.time()-tiempo_inicio:.1f}s")

  # CASO BASE
  if tribunales_asignados >= max_tribunales:
    desbalance = calcular_desbalance()
    if desbalance < mejor_desbalance:
      mejor_desbalance = desbalance
      mejor_solucion = [t.copy() for t in solucion_actual]
      print(f"✓ MEJOR: {desbalance:.4f}")
    return

  # PODA: Tiempo límite
  if time.time() - tiempo_inicio > 30:  # 30 segundos
    return

  # PODA: Nodos límite
  if nodos_explorados > 50000:
    return

  # PODA: No quedan suficientes slots
  if idx_slot >= len(slots_buenos):
    return
  slots_restantes = len(slots_buenos) - idx_slot
  if slots_restantes < (max_tribunales - tribunales_asignados):
    return

  dia, hora = slots_buenos[idx_slot]

  disponibles = set(disp[(dia, hora)])
  libres = {p for p in disponibles if not esta_ocupado(p, dia, hora)}

  cand_P = libres & presidentes_set
  cand_S = libres & secretarios_set
  cand_V = libres & vocales_set

  # Usar slot
  if cand_P and cand_S and cand_V:
    # LIMITAR combinaciones (no todas)
    for pres in list(cand_P)[:3]:  # Solo 3 presidentes
      for sec in list(cand_S)[:3]:  # Solo 3 secretarios
        if sec == pres:
          continue
        for voc in list(cand_V)[:3]:  # Solo 3 vocales
          if voc in [pres, sec]:
            continue

          asignar_tribunal(tribunales_asignados, dia, hora, pres, sec, voc)
          generar_tribunales(idx_slot + 1, tribunales_asignados + 1)
          deshacer_tribunal()

  # Saltar slot
  generar_tribunales(idx_slot + 1, tribunales_asignados)

print("INICIANDO...")
generar_tribunales(0, 0)

print("\n" + "="*60)
if mejor_solucion:
  print(f"MEJOR DESBALANCE: {mejor_desbalance:.4f}")
  print(f"Nodos explorados: {nodos_explorados}")
  print(f"Tiempo: {time.time()-tiempo_inicio:.2f}s")
  print(mejor_solucion)
else:
  print("No se encontró solución en el tiempo límite")

In [ ]:
import time

max_tribunales = 15

presidentes_set = set([p for p in profesores if 'P' in roles[p]])
secretarios_set = set([p for p in profesores if 'S' in roles[p]])
vocales_set = set([p for p in profesores if 'V' in roles[p]])

todos_slots = [(d, h) for d in dias for h in horas]

# ORDENAR POR DISPONIBILIDAD
slots_con_disponibilidad = []
for slot in todos_slots:
  dia, hora = slot
  disponibles = len(disp[(dia, hora)])
  slots_con_disponibilidad.append((slot, disponibles))

# Ordenar de mayor a menor disponibilidad
slots_con_disponibilidad.sort(key=lambda x: x[1], reverse=True)

# Tomar los 15 con más disponibilidad
slots_seleccionados = [slot for slot, _ in slots_con_disponibilidad[:max_tribunales]]

print(f"Slots seleccionados (por disponibilidad):")
for slot in slots_seleccionados:
  print(f"  {slot}: {len(disp[slot])} disponibles")

# Variables globales
mejor_solucion = None
mejor_desbalance = float('inf')
solucion_actual = []
ocupados = {}
cargas = {p: 0 for p in profesores}
tiempo_inicio = time.time()

def esta_ocupado(profesor, dia, hora):
  return (dia, hora) in ocupados and profesor in ocupados[(dia, hora)]

def asignar_tribunal(n, dia, hora, pres, sec, voc):
  solucion_actual.append({'tribunal': n, 'dia': dia, 'hora': hora, 'P': pres, 'S': sec, 'V': voc})
  if (dia, hora) not in ocupados:
    ocupados[(dia, hora)] = []
  for prof in [pres, sec, voc]:
    ocupados[(dia, hora)].append(prof)
    cargas[prof] += 1

def deshacer_tribunal():
  if not solucion_actual:
    return
  t = solucion_actual.pop()
  for prof in [t['P'], t['S'], t['V']]:
    cargas[prof] -= 1
    ocupados[(t['dia'], t['hora'])].remove(prof)
  if not ocupados[(t['dia'], t['hora'])]:
    del ocupados[(t['dia'], t['hora'])]

def calcular_desbalance():
  valores = list(cargas.values())
  promedio = sum(valores) / len(valores)
  varianza = sum((c - promedio)**2 for c in valores) / len(valores)
  return varianza ** 0.5

def calcular_cota_inferior():
  tribunales_asignados = len(solucion_actual)
  tribunales_restantes = max_tribunales - tribunales_asignados

  if tribunales_restantes == 0:
    return calcular_desbalance()

  asignaciones_restantes = tribunales_restantes * 3
  cargas_proyectadas = cargas.copy()

  for _ in range(asignaciones_restantes):
    profesor_menos_cargado = min(cargas_proyectadas, key=cargas_proyectadas.get)
    cargas_proyectadas[profesor_menos_cargado] += 1

  valores = list(cargas_proyectadas.values())
  promedio = sum(valores) / len(valores)
  varianza = sum((c - promedio)**2 for c in valores) / len(valores)
  return varianza ** 0.5

def generar_tribunales(idx_slot):
  global mejor_solucion, mejor_desbalance

  if idx_slot >= len(slots_seleccionados):
    desbalance = calcular_desbalance()
    if desbalance < mejor_desbalance:
      mejor_desbalance = desbalance
      mejor_solucion = [t.copy() for t in solucion_actual]
      print(f"✓ Mejor: {desbalance:.4f}")
    return

  # Poda por cota
  cota = calcular_cota_inferior()
  if cota >= mejor_desbalance:
    return

  # Límite de tiempo
  if time.time() - tiempo_inicio > 120:
    return

  dia, hora = slots_seleccionados[idx_slot]
  disponibles = set(disp[(dia, hora)])
  libres = {p for p in disponibles if not esta_ocupado(p, dia, hora)}

  cand_P = libres & presidentes_set
  cand_S = libres & secretarios_set
  cand_V = libres & vocales_set

  if cand_P and cand_S and cand_V:
    # Ordenar por carga
    cand_P_ordenados = sorted(cand_P, key=lambda p: cargas[p])
    cand_S_ordenados = sorted(cand_S, key=lambda p: cargas[p])
    cand_V_ordenados = sorted(cand_V, key=lambda p: cargas[p])

    for pres in cand_P_ordenados:
      for sec in cand_S_ordenados:
        if sec == pres:
          continue
        for voc in cand_V_ordenados:
          if voc in [pres, sec]:
            continue

          asignar_tribunal(idx_slot, dia, hora, pres, sec, voc)
          generar_tribunales(idx_slot + 1)
          deshacer_tribunal()

print("\nIniciando Ramificación y Poda...")
generar_tribunales(0)

print("\n" + "="*60)
print("RESULTADO FINAL")
print("="*60)
if mejor_solucion:
  print(f"Desbalance: {mejor_desbalance:.4f}")

  cargas_finales = {p: 0 for p in profesores}
  for t in mejor_solucion:
    cargas_finales[t['P']] += 1
    cargas_finales[t['S']] += 1
    cargas_finales[t['V']] += 1

  print("\nCargas:")
  for prof, carga in sorted(cargas_finales.items(), key=lambda x: x[1], reverse=True):
    print(f"  {prof}: {carga}")

  print("\nTribunales:")
  for t in sorted(mejor_solucion, key=lambda x: x['tribunal']):
    print(f"  T{t['tribunal']}: Día {t['dia']}, {t['hora']}:00 - P:{t['P']}, S:{t['S']}, V:{t['V']}")
else:
  print("No se encontró solución")

# Uso de IA

La IA se usó inicialmente para generar el diccionario de disponibilidad de profesores, ya que era muy extenso. Necesité consultar qué concepto matemático usar para medir el reparto de cargas para poder plantear la solución. También se usó para consultar si el planteamiento era cohoerente, y constultar posibles cambios de planteamiento y si mejoraba su eficiencia. Dejo a continuación conversaciones con la IA.

Porfa, crea un diccionario con cada profesor y la disponibilidad que tienen para los días y horas especificadas, con un formato como este:

disponibilidad = {
  'RRD': {
    (15, 15): 0, (15, 16): 1, (15, 17): 1, (15, 18): 1, (15, 19): 0, (15, 20): 1, (15, 21): 1, etc
...
  }
}
adjunto imagen para que veas la información

In [ ]:
'''
Viewing the availability matrix image to extract the data for all professors
Perfecto, veo la matriz completa. Te creo el diccionario con todos los datos:

disponibilidad = {
    'RRD': {
        (15, 15): 0, (15, 16): 1, (15, 17): 1, (15, 18): 1, (15, 19): 0, (15, 20): 1, (15, 21): 1,
        (16, 15): 1, (16, 16): 0, (16, 17): 1, (16, 18): 1, (16, 19): 1, (16, 20): 1, (16, 21): 1,
        (17, 15): 1, (17, 16): 1, (17, 17): 0, (17, 18): 0, (17, 19): 1, (17, 20): 0, (17, 21): 1,
        (18, 15): 0, (18, 16): 1, (18, 17): 1, (18, 18): 1, (18, 19): 1, (18, 20): 1, (18, 21): 1,
        (19, 15): 1, (19, 16): 1, (19, 17): 1, (19, 18): 1, (19, 19): 1, (19, 20): 0, (19, 21): 0,
    },
    'QYV': {
        (15, 15): 1, (15, 16): 1, (15, 17): 1, (15, 18): 1, (15, 19): 0, (15, 20): 0, (15, 21): 0,
        (16, 15): 0, (16, 16): 1, (16, 17): 1, (16, 18): 1, (16, 19): 1, (16, 20): 0, (16, 21): 0,
        (17, 15): 1, (17, 16): 1, (17, 17): 0, (17, 18): 0, (17, 19): 1, (17, 20): 1, (17, 21): 1,
        (18, 15): 0, (18, 16): 1, (18, 17): 1, (18, 18): 1, (18, 19): 1, (18, 20): 1, (18, 21): 1,
        (19, 15): 1, (19, 16): 1, (19, 17): 1, (19, 18): 1, (19, 19): 1, (19, 20): 1, (19, 21): 1,
    },
    'LJL': {
        (15, 15): 1, (15, 16): 0, (15, 17): 0, (15, 18): 1, (15, 19): 1, (15, 20): 0, (15, 21): 1,
        (16, 15): 1, (16, 16): 1, (16, 17): 1, (16, 18): 1, (16, 19): 1, (16, 20): 0, (16, 21): 0,
        (17, 15): 1, (17, 16): 1, (17, 17): 1, (17, 18): 1, (17, 19): 1, (17, 20): 1, (17, 21): 1,
        (18, 15): 1, (18, 16): 1, (18, 17): 1, (18, 18): 0, (18, 19): 1, (18, 20): 1, (18, 21): 1,
        (19, 15): 0, (19, 16): 1, (19, 17): 1, (19, 18): 1, (19, 19): 0, (19, 20): 1, (19, 21): 0,
    },
    'HLC': {
        (15, 15): 1, (15, 16): 0, (15, 17): 1, (15, 18): 1, (15, 19): 0, (15, 20): 1, (15, 21): 1,
        (16, 15): 0, (16, 16): 1, (16, 17): 1, (16, 18): 0, (16, 19): 0, (16, 20): 1, (16, 21): 1,
        (17, 15): 1, (17, 16): 0, (17, 17): 0, (17, 18): 1, (17, 19): 1, (17, 20): 1, (17, 21): 1,
        (18, 15): 1, (18, 16): 1, (18, 17): 1, (18, 18): 0, (18, 19): 1, (18, 20): 0, (18, 21): 1,
        (19, 15): 1, (19, 16): 1, (19, 17): 1, (19, 18): 1, (19, 19): 1, (19, 20): 1, (19, 21): 0,
    },
    'MSB': {
        (15, 15): 1, (15, 16): 1, (15, 17): 0, (15, 18): 1, (15, 19): 0, (15, 20): 1, (15, 21): 1,
        (16, 15): 1, (16, 16): 1, (16, 17): 1, (16, 18): 0, (16, 19): 1, (16, 20): 1, (16, 21): 1,
        (17, 15): 1, (17, 16): 1, (17, 17): 1, (17, 18): 0, (17, 19): 1, (17, 20): 1, (17, 21): 1,
        (18, 15): 0, (18, 16): 1, (18, 17): 1, (18, 18): 1, (18, 19): 1, (18, 20): 0, (18, 21): 1,
        (19, 15): 1, (19, 16): 1, (19, 17): 1, (19, 18): 0, (19, 19): 1, (19, 20): 1, (19, 21): 0,
    },
    'PMQ': {
        (15, 15): 1, (15, 16): 1, (15, 17): 1, (15, 18): 1, (15, 19): 1, (15, 20): 0, (15, 21): 0,
        (16, 15): 1, (16, 16): 1, (16, 17): 1, (16, 18): 1, (16, 19): 1, (16, 20): 1, (16, 21): 1,
        (17, 15): 1, (17, 16): 1, (17, 17): 1, (17, 18): 0, (17, 19): 0, (17, 20): 1, (17, 21): 1,
        (18, 15): 1, (18, 16): 1, (18, 17): 1, (18, 18): 1, (18, 19): 1, (18, 20): 0, (18, 21): 0,
        (19, 15): 1, (19, 16): 1, (19, 17): 1, (19, 18): 1, (19, 19): 0, (19, 20): 1, (19, 21): 0,
    },
    'QWF': {
        (15, 15): 0, (15, 16): 1, (15, 17): 1, (15, 18): 1, (15, 19): 1, (15, 20): 1, (15, 21): 1,
        (16, 15): 1, (16, 16): 1, (16, 17): 0, (16, 18): 1, (16, 19): 1, (16, 20): 0, (16, 21): 1,
        (17, 15): 0, (17, 16): 0, (17, 17): 1, (17, 18): 1, (17, 19): 1, (17, 20): 0, (17, 21): 0,
        (18, 15): 1, (18, 16): 1, (18, 17): 1, (18, 18): 0, (18, 19): 0, (18, 20): 0, (18, 21): 0,
        (19, 15): 1, (19, 16): 1, (19, 17): 1, (19, 18): 1, (19, 19): 1, (19, 20): 0, (19, 21): 1,
    },
    'EBB': {
        (15, 15): 1, (15, 16): 1, (15, 17): 1, (15, 18): 1, (15, 19): 1, (15, 20): 0, (15, 21): 0,
        (16, 15): 1, (16, 16): 1, (16, 17): 0, (16, 18): 1, (16, 19): 1, (16, 20): 1, (16, 21): 1,
        (17, 15): 0, (17, 16): 1, (17, 17): 1, (17, 18): 1, (17, 19): 0, (17, 20): 0, (17, 21): 1,
        (18, 15): 1, (18, 16): 1, (18, 17): 0, (18, 18): 1, (18, 19): 1, (18, 20): 1, (18, 21): 1,
        (19, 15): 1, (19, 16): 0, (19, 17): 1, (19, 18): 1, (19, 19): 1, (19, 20): 1, (19, 21): 0,
    },
    'IOE': {
        (15, 15): 1, (15, 16): 0, (15, 17): 1, (15, 18): 1, (15, 19): 0, (15, 20): 1, (15, 21): 0,
        (16, 15): 0, (16, 16): 1, (16, 17): 1, (16, 18): 1, (16, 19): 1, (16, 20): 1, (16, 21): 1,
        (17, 15): 1, (17, 16): 1, (17, 17): 1, (17, 18): 0, (17, 19): 0, (17, 20): 0, (17, 21): 1,
        (18, 15): 1, (18, 16): 1, (18, 17): 1, (18, 18): 1, (18, 19): 1, (18, 20): 1, (18, 21): 1,
        (19, 15): 1, (19, 16): 0, (19, 17): 1, (19, 18): 0, (19, 19): 1, (19, 20): 1, (19, 21): 1,
    },
    'IOA': {
        (15, 15): 1, (15, 16): 1, (15, 17): 0, (15, 18): 1, (15, 19): 1, (15, 20): 0, (15, 21): 1,
        (16, 15): 1, (16, 16): 0, (16, 17): 0, (16, 18): 0, (16, 19): 0, (16, 20): 0, (16, 21): 1,
        (17, 15): 1, (17, 16): 1, (17, 17): 1, (17, 18): 0, (17, 19): 0, (17, 20): 1, (17, 21): 1,
        (18, 15): 1, (18, 16): 0, (18, 17): 0, (18, 18): 0, (18, 19): 0, (18, 20): 1, (18, 21): 1,
        (19, 15): 1, (19, 16): 1, (19, 17): 1, (19, 18): 1, (19, 19): 0, (19, 20): 0, (19, 21): 0,
    },
}

#Listo! Ahora puedes usar el diccionario así:
# Verificar si RRD está disponible el día 15 a las 16h
if disponibilidad['RRD'][(15, 16)] == 1:
    print("RRD disponible")

# Verificar si QYV está disponible el día 18 a las 20h
if disponibilidad['QYV'][(18, 20)] == 1:
    print("QYV disponible")
'''

tengo un grupo de 15 profesores a los que tengo que asignar un número total de tribunales.  se deben configurar 15 tribunales buscando la configuración
más equilibrada posible en cuanto a la cantidad de tribunales asignados a cada profesor, es
decir, evitando que un profesor tenga muchos tribunales y otros pocos.

cómo puedo calcular matemáticamente este reparto de carga?

In [ ]:
'''Para calcular el reparto más equilibrado de 15 tribunales entre 15 profesores, puedes usar este enfoque matemático:
Cálculo básico
Carga promedio = Total de tribunales / Total de profesores = 15/15 = 1 tribunal por profesor
En este caso específico, la solución es directa: cada profesor debería tener exactamente 1 tribunal asignado.
Fórmula general (para cualquier caso)
Para casos donde los números no dividen exactamente, usa:

Carga base: q = ⌊n/p⌋ (división entera)

donde n = número de tribunales, p = número de profesores


Profesores con carga extra: r = n mod p (resto de la división)

r profesores tendrán (q+1) tribunales
(p-r) profesores tendrán q tribunales


Desviación estándar (para medir equilibrio):

   σ = √[Σ(xi - x̄)² / p]
donde xi es la carga de cada profesor y x̄ es la media'''

'''def generar_tribunal(n, max, profesores, dias, horas, roles, solucion):
  solucion.append({
      'tribunal': n
  })
  if n == max:
    return solucion
  return generar_tribunal(n+1, max, profesores, dias, horas, roles, solucion)

def start(max, profesores, dias, horas, roles):
  solucion = []
  
  return generar_tribunal(1, max, profesores, dias, horas, roles, solucion)


print(start(15, profesores, dias, horas, roles))'''

cómo lo ves para empezar a plantear la solución?

In [ ]:
'''Análisis de tu Enfoque Inicial
Evaluación: 3/10 ⚠️
Tu código tiene la idea correcta (recursión para generar tribunales), pero tiene problemas estructurales importantes.
...
'''


coño te estoy diciendo para EMPEZAR ,  solo quiero hacerme una idea de si la recursividad está bien planteada

In [ ]:
'''¡Ah, vale! Solo para la Recursividad Base
Respuesta Directa: SÍ, está BIEN planteada ✅
Tu recursividad básica es correcta para empezar:
pythondef generar_tribunal(n, max, ...):
    solucion.append({'tribunal': n})

    if n == max:                          # ← Caso base correcto
        return solucion

    return generar_tribunal(n+1, ...)     # ← Llamada recursiva correcta

Lo que está BIEN:
✅ Caso base correcto: if n == max: return
✅ Recursión avanza: n+1 progresa hacia el final
✅ Retorna resultado: return solucion
✅ Estructura clara: Fácil de entender

Pequeño ajuste para que funcione mejor:
pythondef generar_tribunal(n, max, profesores, dias, horas, roles, solucion):
    # Caso base PRIMERO (mejor práctica)
    if n > max:  # ← Cambio: > en vez de ==
        return solucion

    solucion.append({
        'tribunal': n,
        # Aquí luego añadirás: 'dia': ..., 'hora': ..., etc.
    })

    return generar_tribunal(n+1, max, profesores, dias, horas, roles, solucion)'''

podría ser más eficiente mirar primero días y horas, y hacer listado de los profesores disponbiels por rol que ir profe por profe para cada rol?

In [ ]:
'''
¿Invertir la Lógica: Días/Horas → Profesores?Respuesta Directa: SÍ, podría ser MÁS eficiente ✅
Análisis de Eficiencia
Costo de Pre-cálculo (una sola vez):
python# Por cada slot (15 slots):
# - Verificar 7 candidatos_P
# - Verificar 9 candidatos_S
# - Verificar 10 candidatos_V

# Total: 15 × (7 + 9 + 10) = 15 × 26 = 390 verificaciones
# Se hace UNA SOLA VEZ al inicio
Costo por Tribunal:
python# Enfoque A (profesor → día):
# 700 verificaciones por tribunal × 15 tribunales = 10,500 verificaciones

# Enfoque B (día → profesor):
# 0 verificaciones por tribunal (ya pre-calculado)
# Solo costo inicial: 390 verificaciones

# GANANCIA: 10,500 → 390 = 96.3% menos verificaciones
'''

usar sets sería más eficiente que usar for?

In [ ]:
'''
¿Sets vs For Loops?Respuesta Directa: Depende de QUÉ estés haciendo ⚠️No es "Sets vs For" - Ambos se Usan Juntospython# MALENTENDIDO COMÚN:
# "¿Uso sets O uso for?"

# REALIDAD:
# Sets y for loops hacen cosas DIFERENTES
# Normalmente usas AMBOS juntosCaso 1: Intersección (Encontrar Elementos Comunes)Con Listas + For Loop:pythondisponibles = ['QYV', 'HLC', 'MSB', 'PMQ', 'EBB', 'IOE', 'IOA']
presidentes = ['RRD', 'QYV', 'LJL', 'MSB', 'PMQ', 'IOE', 'IOA']

# Encontrar quién está disponible Y puede ser presidente
candidatos_P = []
for p in disponibles:           # 7 iteraciones
    if p in presidentes:        # Cada "in" busca en lista → O(7)
        candidatos_P.append(p)

# Complejidad: O(7 × 7) = O(49) operacionesCon Sets (sin for explícito):pythondisponibles_set = set(disponibles)
presidentes_set = set(presidentes)

# Intersección directa
candidatos_P = disponibles_set & presidentes_set

# Complejidad: O(min(7, 7)) = O(7) operaciones
# 7x MÁS RÁPIDOResultado: Sets ganan ✅Caso 2: Iteración Simple (Procesar Cada Elemento)Con For Loop:pythondisponibles = ['QYV', 'HLC', 'MSB', 'PMQ', 'EBB', 'IOE', 'IOA']

# Imprimir cada profesor
for prof in disponibles:
    print(prof)

# Complejidad: O(n)Con Set:pythondisponibles_set = {'QYV', 'HLC', 'MSB', 'PMQ', 'EBB', 'IOE', 'IOA'}

# Imprimir cada profesor
for prof in disponibles_set:  # ← Sigues usando for!
    print(prof)

# Complejidad: O(n)Resultado: IGUAL (no hay ventaja) 🟰Caso 3: Búsqueda (Verificar si Existe)Con Lista:pythonpresidentes = ['RRD', 'QYV', 'LJL', 'MSB', 'PMQ', 'IOE', 'IOA']

# Verificar si 'MSB' puede ser presidente
if 'MSB' in presidentes:  # Busca secuencialmente
    print("Puede")

# Complejidad: O(n) - puede recorrer toda la lista
# En promedio: O(n/2)Con Set:pythonpresidentes_set = {'RRD', 'QYV', 'LJL', 'MSB', 'PMQ', 'IOE', 'IOA'}

# Verificar si 'MSB' puede ser presidente
if 'MSB' in presidentes_set:  # Lookup en hash table
    print("Puede")

# Complejidad: O(1) - directoResultado: Set gana ✅

Tabla Comparativa
OperaciónLista + ForSetGanadorIntersecciónO(n × m)O(min(n, m))Set 🏆Búsqueda (in)O(n)O(1)Set 🏆Iteración simpleO(n)O(n)Empate 🟰Acceso por índiceO(1)❌ No soportaLista 🏆Mantener orden✅ Sí❌ NoLista 🏆Eliminar duplicadosRequiere código extraAutomáticoSet 🏆
'''